ModuleNotFoundError: No module named 'zakup_serv'

In [ ]:
import asyncio
import time
import aiohttp
from bs4 import BeautifulSoup
from string import Template
import re
import os
from functools import wraps

CONTRACT_NUM_GLOBAL = set()


class NoContractsException(BaseException):
    pass


def check_exec_time(func):
    @wraps(func)
    async def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = await func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"{func.__name__} executed in {end_time - start_time:.4f} seconds")
        return result
    return wrapper



async def fetch_page(session, url):
    async with session.get(url, timeout=10) as response:
        response.raise_for_status()
        page_test = await response.text()
        return page_test


async def extract_contract_numbers(html, page_num):
    soup = BeautifulSoup(html, 'lxml')
    #print(soup.prettify())

    global CONTRACT_NUM_GLOBAL

    contract_links = []

    contracts_blocs = soup.find_all("div", class_="registry-entry__header-mid__number")
    contract_links = [contract.find("a").get("href").strip() for contract in contracts_blocs]
    contract_nums = [re.search(r"reestrNumber=(\d+)", link).group(1) for link in contract_links]

    print(len(contracts_blocs))

    # добавим только уникальные ссылки
    if not contract_nums or len(contract_nums) == 0:
        raise NoContractsException(f"На странице {page_num} не найдено контрактов")

    new_contract_nums = [item for item in contract_nums if item not in CONTRACT_NUM_GLOBAL]
    if len(new_contract_nums) == 0:
        raise NoContractsException(f"На странице {page_num} не найдено новых контрактов")

    CONTRACT_NUM_GLOBAL.update(new_contract_nums)



    # тестовая запись содержимого страницы в файл
    save_dir = "downloaded"

    os.makedirs(save_dir, exist_ok=True)
    file_path = os.path.join(save_dir, f"page_{page_num}.html")

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(soup.prettify())

    return new_contract_nums


async def worker(url, session, semaphore):
    async with semaphore:
        try:
            html = await fetch_page(session, url[1])
            contracts_nums_list = await extract_contract_numbers(html, url[0])
            # contracts_nums_list = "dumb"
            print("hit page...")
            result = {
                "url": url[1],
                "page": url[0],
                "contracts_nums": contracts_nums_list
            }
            #print(result)
            if len(contracts_nums_list) == 0:
                raise NoContractsException(f"На странице {url[0]} не найдено новых контрактов")
            return result

        except Exception as e:
            print(f"Ошибка при обработке URL {url}: {e}")
            raise




def page_interval_generator(start=1):
    shift = 10
    current = start
    while True:
        yield [current, current+shift]
        current += shift


@check_exec_time
async def main(urls: list):
    semaphore = asyncio.Semaphore(10)

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36"
    }

    connector = aiohttp.TCPConnector(ssl=False)
    async with aiohttp.ClientSession(headers=headers, connector=connector) as session:
        tasks = [asyncio.create_task(worker(url, session, semaphore)) for url in urls]

        try:
            results = await asyncio.gather(*tasks, return_exceptions=True)
            ok, errors = {}, {}
            for url, res in zip(urls, results):
                print(res)

                if isinstance(res, Exception):
                    errors[url] = repr(res)
                else:
                    ok = res

                if NoContractsException in res:
                    raise NoContractsException(f"На странице {url[0]} не найдено новых контрактов")

            return ok, errors

        except Exception as e:
            print(f"Обнаружено исключение: {e}")
            # Отмена всех задач
            for task in tasks:
                task.cancel()
            # Дожидаемся завершения отменённых задач
            await asyncio.gather(*tasks, return_exceptions=True)
            print("Все задачи остановлены из-за исключения.")
            raise  # Пробрасываем исключение дальше




if __name__ == '__main__':

    base_url = Template("https://zakupki.gov.ru/epz/contract/search/results.html?morphology=on&fz44=on&contractStageList_0=on&contractStageList_1=on&contractStageList_2=on&contractStageList_3=on&contractStageList=0%2C1%2C2%2C3&selectedContractDataChanges=ANY&budgetLevelsIdNameHidden=%7B%7D&customerPlace=44000000000&contractDateFrom=01.01.2025&executionDateStart=31.12.2025&countryRegIdNameHidden=%7B%7D&sortBy=UPDATE_DATE&pageNumber=$page_number&sortDirection=false&recordsPerPage=_200&showLotsInfoHidden=false")

    #base_url = Template("https://zakupki.gov.ru/epz/contract/search/results.html?searchString=&morphology=on&search-filter=%D0%94%D0%B0%D1%82%D0%B5+%D0%BE%D0%B1%D0%BD%D0%BE%D0%B2%D0%BB%D0%B5%D0%BD%D0%B8%D1%8F&savedSearchSettingsIdHidden=&fz44=on&contractStageList_0=on&contractStageList_1=on&contractStageList_2=on&contractStageList_3=on&contractStageList=0%2C1%2C2%2C3&contractInputNameDefenseOrderNumber=&contractInputNameContractNumber=&contractPriceFrom=100000&rightPriceRurFrom=&priceFromUnitGWS=&contractPriceTo=101000&rightPriceRurTo=&priceToUnitGWS=&currencyCode=&nonBudgetCodesList=&budgetLevelsIdHidden=&budgetLevelsIdNameHidden=%7B%7D&budgetName=&customerPlace=44000000000&customerPlaceCodes=44000000000&contractDateFrom=01.01.2025&contractDateTo=&executionDateStart=31.12.2025&executionDateEnd=&publishDateFrom=&publishDateTo=&updateDateFrom=&updateDateTo=&placingWayList=&selectedLaws=&sortBy=UPDATE_DATE&pageNumber=$page_number&sortDirection=false&recordsPerPage=_10&showLotsInfoHidden=false")



    for pages in page_interval_generator():

        target_urls = [(i, base_url.substitute(page_number=i)) for i in range(pages[0], pages[1])]

        try:
            data, errs = asyncio.run(main(target_urls))
            print("data: ", data)
            print("errors: ", errs)

            print(len(CONTRACT_NUM_GLOBAL))

        except Exception as e:
            print(e)
            break



    print("total ", len(CONTRACT_NUM_GLOBAL))
    with open("contracts.txt", "w") as f:
        f.write("\n".join(CONTRACT_NUM_GLOBAL))